# Iceberg V3 Basics - VARIANT & Nanosecond Timestamps

**Release:** v1.0.0 | **Date:** 2026-03-17

## Test Case: v3 Data Types (Private Preview)
- **Capability**: Semi-structured data in v3 format  
- **Target**: Query performance within 15% of Native tables
- **Features**: VARIANT, TIMESTAMP_NTZ(9), nested JSON

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Test 1: Create Iceberg v3 PATIENT_EVENTS table with VARIANT and nanosecond timestamps
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.PATIENT_EVENTS (
    event_id        INT,
    patient_id      INT,
    event_ts        TIMESTAMP_NTZ(9),
    event_type      VARCHAR,
    clinical_data   VARIANT,
    metadata        VARIANT
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

In [ ]:
-- Test 2: Insert 1M clinical event records with nested VARIANT and nanosecond timestamps
INSERT INTO ICEBERG_POC.TESTS.PATIENT_EVENTS
SELECT
    SEQ4()                                                                                                 AS event_id,
    SEQ4() % 100000 + 1001                                                                                AS patient_id,
    DATEADD('nanosecond', UNIFORM(0, 999999999, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ(9)         AS event_ts,
    ARRAY_CONSTRUCT('ADMISSION','DISCHARGE','LAB_RESULT','MEDICATION_ORDER','VITAL_SIGNS','RADIOLOGY')[SEQ4() % 6]::VARCHAR AS event_type,
    OBJECT_CONSTRUCT(
        'icd10_codes',  ARRAY_CONSTRUCT(
            ARRAY_CONSTRUCT('E11.9','I10','J06.9','M54.5','Z00.00','F32.1')[SEQ4() % 6]::VARCHAR,
            ARRAY_CONSTRUCT('I10','E78.5','K21.0','N39.0','J45.9','M54.5')[SEQ4() % 6]::VARCHAR
        ),
        'vitals',       OBJECT_CONSTRUCT(
            'bp_systolic',  (SEQ4() % 60 + 100)::INT,
            'bp_diastolic', (SEQ4() % 40 + 60)::INT,
            'heart_rate',   (SEQ4() % 60 + 60)::INT,
            'temperature',  ROUND(97.5 + (SEQ4() % 40) / 20.0, 1)
        ),
        'lab_results',  ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('test', 'HbA1c',   'value', ROUND(5.0 + (SEQ4() % 60) / 10.0, 1), 'unit', '%'),
            OBJECT_CONSTRUCT('test', 'Glucose',  'value', (SEQ4() % 150 + 70)::INT,             'unit', 'mg/dL'),
            OBJECT_CONSTRUCT('test', 'Creatinine','value', ROUND(0.5 + (SEQ4() % 20) / 10.0, 1),'unit', 'mg/dL')
        )
    ) AS clinical_data,
    OBJECT_CONSTRUCT(
        'source',       ARRAY_CONSTRUCT('EHR','Lab_System','Pharmacy','Radiology_PACS')[SEQ4() % 4]::VARCHAR,
        'version',      'v3.0',
        'facility_id',  (SEQ4() % 50 + 1)::INT
    ) AS metadata
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

In [ ]:
-- Test 3: Query nested clinical VARIANT data with nanosecond timestamp precision
SELECT
    event_id,
    TO_VARCHAR(event_ts, 'YYYY-MM-DD HH24:MI:SS.FF9')       AS event_ts_nanoseconds,
    event_type,
    clinical_data:icd10_codes[0][0]::VARCHAR                 AS primary_dx_code,
    clinical_data:vitals:bp_systolic::INT                    AS systolic_bp,
    clinical_data:vitals:heart_rate::INT                     AS heart_rate,
    clinical_data:lab_results[0]:test::VARCHAR               AS lab_test,
    clinical_data:lab_results[0]:value::FLOAT                AS lab_value,
    metadata:source::VARCHAR                                  AS source_system
FROM ICEBERG_POC.TESTS.PATIENT_EVENTS
WHERE clinical_data:vitals:bp_systolic::INT > 140
LIMIT 100;

In [ ]:
-- Test 4: Clinical aggregations on VARIANT data
SELECT
    event_type,
    COUNT(*)                                                    AS event_count,
    ROUND(AVG(clinical_data:vitals:bp_systolic::INT), 1)       AS avg_systolic,
    ROUND(AVG(clinical_data:vitals:heart_rate::INT), 1)        AS avg_heart_rate,
    SUM(CASE WHEN clinical_data:vitals:bp_systolic::INT > 140 THEN 1 ELSE 0 END) AS hypertensive_events,
    metadata:source::VARCHAR                                    AS source_system
FROM ICEBERG_POC.TESTS.PATIENT_EVENTS
GROUP BY event_type, metadata:source::VARCHAR
ORDER BY event_count DESC;

In [ ]:
-- Verify PATIENT_EVENTS table metadata
SHOW ICEBERG TABLES LIKE 'PATIENT_EVENTS' IN SCHEMA ICEBERG_POC.TESTS;

SELECT COUNT(*) AS total_rows FROM ICEBERG_POC.TESTS.PATIENT_EVENTS;